# HF Transformers 核心模块学习：Pipelines

**Pipelines**（管道）是使用模型进行推理的一种简单易上手的方式。

这些管道是抽象了 Transformers 库中大部分复杂代码的对象，提供了一个专门用于多种任务的简单API，包括**命名实体识别、掩码语言建模、情感分析、特征提取和问答**等。


| Modality                    | Task                         | Description                                                | Pipeline API                                  |
| --------------------------- | ---------------------------- | ---------------------------------------------------------- | --------------------------------------------- |
| Audio                       | Audio classification         | 为音频文件分配一个标签                                     | pipeline(task=“audio-classification”)         |
|                             | Automatic speech recognition | 将音频文件中的语音提取为文本                               | pipeline(task=“automatic-speech-recognition”) |
| Computer vision             | Image classification         | 为图像分配一个标签                                         | pipeline(task=“image-classification”)         |
|                             | Object detection             | 预测图像中目标对象的边界框和类别                           | pipeline(task=“object-detection”)             |
|                             | Image segmentation           | 为图像中每个独立的像素分配标签（支持语义、全景和实例分割） | pipeline(task=“image-segmentation”)           |
| Natural language processing | Text classification          | 为给定的文本序列分配一个标签                               | pipeline(task=“sentiment-analysis”)           |
|                             | Token classification         | 为序列里的每个 token 分配一个标签（人, 组织, 地址等等）    | pipeline(task=“ner”)                          |
|                             | Question answering           | 通过给定的上下文和问题, 在文本中提取答案                   | pipeline(task=“question-answering”)           |
|                             | Summarization                | 为文本序列或文档生成总结                                   | pipeline(task=“summarization”)                |
|                             | Translation                  | 将文本从一种语言翻译为另一种语言                           | pipeline(task=“translation”)                  |
| Multimodal                  | Document question answering  | 根据给定的文档和问题回答一个关于该文档的问题。             | pipeline(task=“document-question-answering”)  |
|                             | Visual Question Answering    | 给定一个图像和一个问题，正确地回答有关图像的问题           | pipeline(task=“vqa”)                          |



Pipelines 已支持的完整任务列表：https://huggingface.co/docs/transformers/task_summary


## Pipeline API

**Pipeline API** 是对所有其他可用管道的包装。它可以像任何其他管道一样实例化，并且降低AI推理的学习和使用成本。

![](docs/images/pipeline_func.png)

### 使用 Pipeline API 实现 Text Classification 任务


**Text classification**(文本分类)与任何模态中的分类任务一样，文本分类将一个文本序列（可以是句子级别、段落或者整篇文章）标记为预定义的类别集合之一。文本分类有许多实际应用，其中包括：

- 情感分析：根据某种极性（如积极或消极）对文本进行标记，以在政治、金融和市场等领域支持决策制定。
- 内容分类：根据某个主题对文本进行标记，以帮助组织和过滤新闻和社交媒体信息流中的信息（天气、体育、金融等）。


下面以 `Text classification` 中的情感分析任务为例，展示如何使用 Pipeline API。

模型主页：https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english

## transformers 自定义模型下载的路径

在transformers自定义模型下载的路径方法

```python
import os

os.environ['HF_HOME'] = '/Users/guoyingwen/mnt/new_volume/hf'
os.environ['HF_HUB_CACHE'] = '/Users/guoyingwen/mnt/new_volume/hf/hub'
```

In [22]:
import os

os.environ['HF_HOME'] = '/Users/guoyingwen/mnt/hf'
os.environ['HF_HUB_CACHE'] = '/Users/guoyingwen/mnt/hf/hub'

In [23]:
from transformers import pipeline

# 仅指定任务时，使用默认模型（不推荐）
pipe = pipeline(
    "sentiment-analysis",
    model="lxyuan/distilbert-base-multilingual-cased-sentiments-student", 
    device="mps",
    return_all_scores=True)
pipe("今儿上海可真冷啊")

Device set to use mps


KeyboardInterrupt: 

### 测试更多示例

In [ ]:
pipe("我觉得这家店蒜泥白肉的味道一般")

[[{'label': 'positive', 'score': 0.07258126884698868},
  {'label': 'neutral', 'score': 0.6030055284500122},
  {'label': 'negative', 'score': 0.3244132101535797}]]

In [ ]:
# 默认使用的模型 distilbert-base-uncased-finetuned-sst-2-english 
# 并未针对中文做太多训练，中文的文本分类任务表现未必满意
pipe("你学东西真的好快，理论课一讲就明白了")

[[{'label': 'positive', 'score': 0.9461327791213989},
  {'label': 'neutral', 'score': 0.03845958411693573},
  {'label': 'negative', 'score': 0.015407565981149673}]]

In [ ]:
# 替换为英文后，文本分类任务的表现立刻改善
pipe("You learn things really quickly. You understand the theory class as soon as it is taught.")

[[{'label': 'positive', 'score': 0.7639098763465881},
  {'label': 'neutral', 'score': 0.15310533344745636},
  {'label': 'negative', 'score': 0.08298485726118088}]]

In [ ]:
pipe("Today Shanghai is really cold.")

[[{'label': 'positive', 'score': 0.09706174582242966},
  {'label': 'neutral', 'score': 0.12048666924238205},
  {'label': 'negative', 'score': 0.7824515700340271}]]

### 批处理调用模型推理

In [ ]:
text_list = [
    "Today Shanghai is really cold.",
    "I think the taste of the garlic mashed pork in this store is average.",
    "You learn things really quickly. You understand the theory class as soon as it is taught."
]

pipe(text_list)

[[{'label': 'positive', 'score': 0.09706174582242966},
  {'label': 'neutral', 'score': 0.12048666924238205},
  {'label': 'negative', 'score': 0.7824515700340271}],
 [{'label': 'positive', 'score': 0.37246423959732056},
  {'label': 'neutral', 'score': 0.250036358833313},
  {'label': 'negative', 'score': 0.37749940156936646}],
 [{'label': 'positive', 'score': 0.7639098763465881},
  {'label': 'neutral', 'score': 0.15310533344745636},
  {'label': 'negative', 'score': 0.08298485726118088}]]

## 使用 Pipeline API 调用更多预定义任务

## Natural Language Processing(NLP)

**NLP**(自然语言处理)任务是最常见的任务类型之一，因为文本是我们进行交流的一种自然方式。要将文本转换为模型可识别的格式，需要对其进行分词。这意味着将一系列文本划分为单独的单词或子词（标记），然后将这些标记转换为数字。结果就是，您可以将一系列文本表示为一系列数字，并且一旦您拥有了一系列数字，它就可以输入到模型中来解决各种NLP任务！

上面演示的 文本分类任务，以及接下来的标记、问答等任务都属于 NLP 范畴。

### Token Classification

在任何NLP任务中，文本都经过预处理，将文本序列分成单个单词或子词。这些被称为tokens。

**Token Classification**（Token分类）将每个token分配一个来自预定义类别集的标签。

两种常见的 Token 分类是：

- 命名实体识别（NER）：根据实体类别（如组织、人员、位置或日期）对token进行标记。NER在生物医学设置中特别受欢迎，可以标记基因、蛋白质和药物名称。
- 词性标注（POS）：根据其词性（如名词、动词或形容词）对标记进行标记。POS对于帮助翻译系统了解两个相同的单词如何在语法上不同很有用（作为名词的银行与作为动词的银行）。

模型主页：https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english

In [ ]:
from transformers import pipeline

# 使用已微调的中文词性标注模型
pos_tagger = pipeline(
    "token-classification",
    model="ckiplab/albert-base-chinese-pos",  # 支持词性标注的模型
    device="mps"  # 指定使用 MPS 设备
)
preds = pos_tagger("我爱自然语言处理")

preds = [
    {
        "entity": pred["entity"],
        "score": round(pred["score"], 4),
        "index": pred["index"],
        "word": pred["word"],
        "start": pred["start"],
        "end": pred["end"],
    }
    for pred in preds
]
print(*preds, sep="\n")

Device set to use mps


{'entity': 'Nh', 'score': np.float32(0.9999), 'index': 1, 'word': '我', 'start': 0, 'end': 1}
{'entity': 'VL', 'score': np.float32(0.9272), 'index': 2, 'word': '爱', 'start': 1, 'end': 2}
{'entity': 'Na', 'score': np.float32(0.8037), 'index': 3, 'word': '自', 'start': 2, 'end': 3}
{'entity': 'D', 'score': np.float32(0.5481), 'index': 4, 'word': '然', 'start': 3, 'end': 4}
{'entity': 'Na', 'score': np.float32(0.3902), 'index': 5, 'word': '语', 'start': 4, 'end': 5}
{'entity': 'VE', 'score': np.float32(0.9598), 'index': 6, 'word': '言', 'start': 5, 'end': 6}
{'entity': 'VC', 'score': np.float32(0.9974), 'index': 7, 'word': '处', 'start': 6, 'end': 7}
{'entity': 'VC', 'score': np.float32(0.9713), 'index': 8, 'word': '理', 'start': 7, 'end': 8}


#### 合并实体

![](data/image/paddlenlp_pos_tagging.png)

In [ ]:
from paddlenlp import Taskflow

# 初始化词法分析模型（自动包含分词+词性标注）
tag = Taskflow("pos_tagging")  # ‌:ml-citation{ref="1,3" data="citationList"}

# 单文本处理示例
text = "我爱自然语言处理技术"
result = tag(text)
print(result)
# 输出：[{'word': ['我', '爱', '自然语言处理', '技术'], 'tag': ['PN', 'VV', 'NN', 'NN']}]  # ‌:ml-citation{ref="7" data="citationList"}


[('我', 'r'), ('爱', 'v'), ('自然语言处理', 'nz'), ('技术', 'n')]


E0318 22:16:27.502445 3973417024 analysis_config.cc:658] Please compile with MKLDNN first to use MKLDNN


In [ ]:
# 单文本处理示例
text = "我是中国人，我爱我的祖国"
result = tag(text)
print(result)

[('我', 'r'), ('是', 'v'), ('中国', 'LOC'), ('人', 'n'), ('，', 'w'), ('我', 'r'), ('爱', 'v'), ('我的祖国', 'n')]


### Question Answering

**Question Answering**(问答)是另一个token-level的任务，返回一个问题的答案，有时带有上下文（开放领域），有时不带上下文（封闭领域）。每当我们向虚拟助手提出问题时，例如询问一家餐厅是否营业，就会发生这种情况。它还可以提供客户或技术支持，并帮助搜索引擎检索您要求的相关信息。

有两种常见的问答类型：

- 提取式：给定一个问题和一些上下文，模型必须从上下文中提取出一段文字作为答案
- 生成式：给定一个问题和一些上下文，答案是根据上下文生成的；这种方法由`Text2TextGenerationPipeline`处理，而不是下面展示的`QuestionAnsweringPipeline`

模型主页：https://huggingface.co/distilbert-base-cased-distilled-squad

In [ ]:
from transformers import pipeline

question_answerer = pipeline("question-answering", "deepset/roberta-base-squad2")

Error while downloading from https://cdn-lfs.hf.co/deepset/roberta-base-squad2/ac5db66fdcfecb400345d09787b71009d60805ef9883451071669cf951b5e2c7?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&Expires=1742314190&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0MjMxNDE5MH19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5oZi5jby9kZWVwc2V0L3JvYmVydGEtYmFzZS1zcXVhZDIvYWM1ZGI2NmZkY2ZlY2I0MDAzNDVkMDk3ODdiNzEwMDlkNjA4MDVlZjk4ODM0NTEwNzE2NjljZjk1MWI1ZTJjNz9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSoifV19&Signature=JFWRweNKm%7EWktzwCFg8KNEhh0fItCURm8lxKu5k4uLOzmmDi818cLhDxjRH4hU0sJnPFPMpBLwzXht6BpVp9srqDglhQY0Eu5iTFIWDzTV0CjhShywyOM5RhFSplWJxtgyz7-5ocDd40iH0ybJk9acwFuVH7U0gUyPA20g1R7YbSfV%7EtC68rp5khfxEF2ZJdmMbAAbe6Uq1HStiFceI94D08VS1tC0ceny-8xVQfMTREoyFLaH6yRl6ddyr7mk1zYaLp2Pgo0OUO6gH%7EeAYS8O6lDk0mgRjWvYs-tSRLKaKoFfiK62X-0zdcr7hCbEJ%7ETmXCdQUS4d%7EcLBHwVUijdg__&Key-Pair-Id=K3RPWS32NSSJCE: HTT

In [ ]:
preds = question_answerer(
    question="What is the name of the repository?",
    context="The name of the repository is huggingface/transformers",
)
print(
    f"score: {round(preds['score'], 4)}, start: {preds['start']}, end: {preds['end']}, answer: {preds['answer']}"
)

score: 0.9068, start: 30, end: 54, answer: huggingface/transformers


In [ ]:
preds = question_answerer(
    question="What is the capital of China?",
    context="On 1 October 1949, CCP Chairman Mao Zedong formally proclaimed the People's Republic of China in Tiananmen Square, Beijing.",
)
print(
    f"score: {round(preds['score'], 4)}, start: {preds['start']}, end: {preds['end']}, answer: {preds['answer']}"
)

score: 0.7504, start: 115, end: 122, answer: Beijing


### Summarization

**Summarization**(文本摘要）从较长的文本中创建一个较短的版本，同时尽可能保留原始文档的大部分含义。摘要是一个序列到序列的任务；它输出比输入更短的文本序列。有许多长篇文档可以进行摘要，以帮助读者快速了解主要要点。法案、法律和财务文件、专利和科学论文等文档可以摘要，以节省读者的时间并作为阅读辅助工具。

与问答类似，摘要有两种类型：

- 提取式：从原始文本中识别和提取最重要的句子
- 生成式：从原始文本中生成目标摘要（可能包括输入文件中没有的新单词）；`SummarizationPipeline`使用生成式方法

模型主页：https://huggingface.co/t5-base

In [ ]:
from transformers import pipeline

# summarizer = pipeline(task="summarization",
#                       model="t5-base",
#                       min_length=8,
#                       max_length=32,
# )
from transformers import pipeline

summarizer = pipeline("summarization", model="Falconsai/text_summarization")


Error while downloading from https://cdn-lfs-us-1.hf.co/repos/9d/9e/9d9e6b97af31f141e2c5101961a1e9aeedb560a93a94359486e71d26f285372f/c2fa71c28e7fe875fd5cf0d1d39737d326db9f2c800acdd637d5528ada3a711d?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&Expires=1742314157&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0MjMxNDE1N319LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy11cy0xLmhmLmNvL3JlcG9zLzlkLzllLzlkOWU2Yjk3YWYzMWYxNDFlMmM1MTAxOTYxYTFlOWFlZWRiNTYwYTkzYTk0MzU5NDg2ZTcxZDI2ZjI4NTM3MmYvYzJmYTcxYzI4ZTdmZTg3NWZkNWNmMGQxZDM5NzM3ZDMyNmRiOWYyYzgwMGFjZGQ2MzdkNTUyOGFkYTNhNzExZD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSoifV19&Signature=VgMRY%7EHu%7E2HXLJt8uGBm8fgDSdiFxJNeF3etRbii64EFVZMK2nlOuhMemn4dH-1qXyf1pg1uc7feph0NMZouYXXNNWltg0WcQWtr7Zd6-WOulQl5tQ6vYO7DlX5xgn0UW-Qp5KG0O0YCGQe48uJ0uU-Nbl6IMJVoB9J98CnBOrVOgGyUQ98RdCi9FjFMrAwipVx0yYgjjDhBCzpqVSEUkJsNpFbPbn%7EBKi3oEQyenwnbvOjcnccNMrTHqQay2x

[{'summary_text': 'Hugging Face has emerged as a prominent and innovative force in NLP . From its inception to its role in democratizing AI, the company has left an indelible mark on the industry . The name "Hugging Face" was chosen to reflect the company\'s mission of making AI models more accessible and friendly to humans .'}]


In [ ]:

ARTICLE = """ 
Hugging Face: Revolutionizing Natural Language Processing
Introduction
In the rapidly evolving field of Natural Language Processing (NLP), Hugging Face has emerged as a prominent and innovative force. This article will explore the story and significance of Hugging Face, a company that has made remarkable contributions to NLP and AI as a whole. From its inception to its role in democratizing AI, Hugging Face has left an indelible mark on the industry.
The Birth of Hugging Face
Hugging Face was founded in 2016 by Clément Delangue, Julien Chaumond, and Thomas Wolf. The name "Hugging Face" was chosen to reflect the company's mission of making AI models more accessible and friendly to humans, much like a comforting hug. Initially, they began as a chatbot company but later shifted their focus to NLP, driven by their belief in the transformative potential of this technology.
Transformative Innovations
Hugging Face is best known for its open-source contributions, particularly the "Transformers" library. This library has become the de facto standard for NLP and enables researchers, developers, and organizations to easily access and utilize state-of-the-art pre-trained language models, such as BERT, GPT-3, and more. These models have countless applications, from chatbots and virtual assistants to language translation and sentiment analysis.
Key Contributions:
1. **Transformers Library:** The Transformers library provides a unified interface for more than 50 pre-trained models, simplifying the development of NLP applications. It allows users to fine-tune these models for specific tasks, making it accessible to a wider audience.
2. **Model Hub:** Hugging Face's Model Hub is a treasure trove of pre-trained models, making it simple for anyone to access, experiment with, and fine-tune models. Researchers and developers around the world can collaborate and share their models through this platform.
3. **Hugging Face Transformers Community:** Hugging Face has fostered a vibrant online community where developers, researchers, and AI enthusiasts can share their knowledge, code, and insights. This collaborative spirit has accelerated the growth of NLP.
Democratizing AI
Hugging Face's most significant impact has been the democratization of AI and NLP. Their commitment to open-source development has made powerful AI models accessible to individuals, startups, and established organizations. This approach contrasts with the traditional proprietary AI model market, which often limits access to those with substantial resources.
By providing open-source models and tools, Hugging Face has empowered a diverse array of users to innovate and create their own NLP applications. This shift has fostered inclusivity, allowing a broader range of voices to contribute to AI research and development.
Industry Adoption
The success and impact of Hugging Face are evident in its widespread adoption. Numerous companies and institutions, from startups to tech giants, leverage Hugging Face's technology for their AI applications. This includes industries as varied as healthcare, finance, and entertainment, showcasing the versatility of NLP and Hugging Face's contributions.
Future Directions
Hugging Face's journey is far from over. As of my last knowledge update in September 2021, the company was actively pursuing research into ethical AI, bias reduction in models, and more. Given their track record of innovation and commitment to the AI community, it is likely that they will continue to lead in ethical AI development and promote responsible use of NLP technologies.
Conclusion
Hugging Face's story is one of transformation, collaboration, and empowerment. Their open-source contributions have reshaped the NLP landscape and democratized access to AI. As they continue to push the boundaries of AI research, we can expect Hugging Face to remain at the forefront of innovation, contributing to a more inclusive and ethical AI future. Their journey reminds us that the power of open-source collaboration can lead to groundbreaking advancements in technology and bring AI within the reach of many.
"""
summarizer(ARTICLE, max_length=800, min_length=30, do_sample=False)

[{'summary_text': 'Hugging Face has emerged as a prominent and innovative force in NLP . From its inception to its role in democratizing AI, the company has left an indelible mark on the industry . The name "Hugging Face" was chosen to reflect the company\'s mission of making AI models more accessible and friendly to humans .'}]

In [ ]:
summarizer(
    """
    In this work, we presented the Transformer, the first sequence transduction model based entirely on attention, 
    replacing the recurrent layers most commonly used in encoder-decoder architectures with multi-headed self-attention. 
    For translation tasks, the Transformer can be trained significantly faster than architectures based on recurrent or convolutional layers. 
    On both WMT 2014 English-to-German and WMT 2014 English-to-French translation tasks, we achieve a new state of the art. 
    In the former task our best model outperforms even all previously reported ensembles.
    """,max_length=1000, min_length=30, do_sample=False
)


Your max_length is set to 1000, but your input_length is only 128. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=64)


[{'summary_text': 'Transformer is the first sequence transduction model based entirely on attention . For translation tasks, the Transformer can be trained significantly faster than architectures based on recurrent or convolutional layers .'}]

In [ ]:
summarizer(
    '''
    Large language models (LLM) are very large deep learning models that are pre-trained on vast amounts of data. 
    The underlying transformer is a set of neural networks that consist of an encoder and a decoder with self-attention capabilities. 
    The encoder and decoder extract meanings from a sequence of text and understand the relationships between words and phrases in it.
    Transformer LLMs are capable of unsupervised training, although a more precise explanation is that transformers perform self-learning. 
    It is through this process that transformers learn to understand basic grammar, languages, and knowledge.
    Unlike earlier recurrent neural networks (RNN) that sequentially process inputs, transformers process entire sequences in parallel. 
    This allows the data scientists to use GPUs for training transformer-based LLMs, significantly reducing the training time.
    '''
)


Your max_length is set to 200, but your input_length is only 182. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=91)


[{'summary_text': 'Large language models (LLM) are very large deep learning models that are pre-trained on vast amounts of data . The underlying transformer is a set of neural networks that consist of an encoder and a decoder with self-attention capabilities . Transformer LLMs are capable of unsupervised training .'}]


## Audio 音频处理任务

音频和语音处理任务与其他模态略有不同，主要是因为音频作为输入是一个连续的信号。与文本不同，原始音频波形不能像句子可以被划分为单词那样被整齐地分割成离散的块。为了解决这个问题，通常在固定的时间间隔内对原始音频信号进行采样。如果在每个时间间隔内采样更多样本，采样率就会更高，音频更接近原始音频源。

以前的方法是预处理音频以从中提取有用的特征。现在更常见的做法是直接将原始音频波形输入到特征编码器中，以提取音频表示。这样可以简化预处理步骤，并允许模型学习最重要的特征。

### Audio classification

**Audio classification**(音频分类)是一项将音频数据从预定义的类别集合中进行标记的任务。这是一个广泛的类别，具有许多具体的应用，其中一些包括：

- 声学场景分类：使用场景标签（“办公室”、“海滩”、“体育场”）对音频进行标记。
- 声学事件检测：使用声音事件标签（“汽车喇叭声”、“鲸鱼叫声”、“玻璃破碎声”）对音频进行标记。
- 标记：对包含多种声音的音频进行标记（鸟鸣、会议中的说话人识别）。
- 音乐分类：使用流派标签（“金属”、“嘻哈”、“乡村”）对音乐进行标记。

模型主页：https://huggingface.co/superb/hubert-base-superb-er

数据集主页：https://huggingface.co/datasets/superb#er

```
情感识别（ER）为每个话语预测一个情感类别。我们采用了最广泛使用的ER数据集IEMOCAP，并遵循传统的评估协议：我们删除不平衡的情感类别，只保留最后四个具有相似数量数据点的类别，并在标准分割的五折交叉验证上进行评估。评估指标是准确率（ACC）。
```

#### 前置依赖包安装

建议在命令行安装必要的音频数据处理包: ffmpeg

```shell
$apt update & apt upgrade
$apt install -y ffmpeg
$pip install ffmpeg ffmpeg-python
```

In [31]:
# classifier = pipeline(task="audio-classification", model="superb/hubert-base-superb-er")
from datasets import load_dataset
from transformers import pipeline

dataset = load_dataset("anton-l/superb_demo", "er", split="session1")

classifier = pipeline("audio-classification", model="superb/wav2vec2-base-superb-er")
labels = classifier(dataset[0]["file"], top_k=5)

Repo card metadata block was not found. Setting CardData to empty.
[2025-03-18 23:50:15,006] [ WARNING] repocard.py:108 - Repo card metadata block was not found. Setting CardData to empty.
Device set to use mps:0


In [39]:
labels = classifier(dataset[4]["file"], top_k=5)
labels

[{'score': 0.8031607270240784, 'label': 'ang'},
 {'score': 0.15327586233615875, 'label': 'hap'},
 {'score': 0.032175514847040176, 'label': 'sad'},
 {'score': 0.011387879028916359, 'label': 'neu'}]

In [33]:
# 使用 Hugging Face Datasets 上的测试文件
preds = classifier("https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac")
preds = [{"score": round(pred["score"], 4), "label": pred["label"]} for pred in preds]
preds

[{'score': 0.8077, 'label': 'sad'},
 {'score': 0.1082, 'label': 'neu'},
 {'score': 0.08, 'label': 'hap'},
 {'score': 0.004, 'label': 'ang'}]

In [40]:
# 使用本地的音频文件做测试
preds = classifier("data/audio/mlk.flac")
preds = [{"score": round(pred["score"], 4), "label": pred["label"]} for pred in preds]
preds

[{'score': 0.8077, 'label': 'sad'},
 {'score': 0.1082, 'label': 'neu'},
 {'score': 0.08, 'label': 'hap'},
 {'score': 0.004, 'label': 'ang'}]

### Automatic speech recognition（ASR）

**Automatic speech recognition**（自动语音识别）将语音转录为文本。这是最常见的音频任务之一，部分原因是因为语音是人类交流的自然形式。如今，ASR系统嵌入在智能技术产品中，如扬声器、电话和汽车。我们可以要求虚拟助手播放音乐、设置提醒和告诉我们天气。

但是，Transformer架构帮助解决的一个关键挑战是低资源语言。通过在大量语音数据上进行预训练，仅在一个低资源语言的一小时标记语音数据上进行微调，仍然可以产生与以前在100倍更多标记数据上训练的ASR系统相比高质量的结果。

模型主页：https://huggingface.co/openai/whisper-small

下面展示使用 `OpenAI Whisper Small` 模型实现 ASR 的 Pipeline API 示例：

In [46]:
from transformers import pipeline

# device = "cuda:0" if torch.cuda.is_available() else "cpu"
# 使用 `model` 参数指定模型
transcriber = pipeline(task="automatic-speech-recognition", model="openai/whisper-base"
                       ,chunk_length_s=30
                    #    ,device=device,
                       )

Device set to use mps:0


In [47]:
text = transcriber("data/audio/mlk.flac")
text

/opt/anaconda3/envs/acp-learn/lib/python3.10/site-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


{'text': ' I have a dream that one day this nation will rise up and live out the true meaning of its creed.'}

## Computer Vision 计算机视觉

**Computer Vision**（计算机视觉）任务中最早成功之一是使用卷积神经网络（CNN）识别邮政编码数字图像。图像由像素组成，每个像素都有一个数值。这使得将图像表示为像素值矩阵变得容易。每个像素值组合描述了图像的颜色。

计算机视觉任务可以通过以下两种通用方式解决：

- 使用卷积来学习图像的层次特征，从低级特征到高级抽象特征。
- 将图像分成块，并使用Transformer逐步学习每个图像块如何相互关联以形成图像。与CNN偏好的自底向上方法不同，这种方法有点像从一个模糊的图像开始，然后逐渐将其聚焦清晰。

### Image Classificaiton

**Image Classificaiton**(图像分类)将整个图像从预定义的类别集合中进行标记。像大多数分类任务一样，图像分类有许多实际用例，其中一些包括：

- 医疗保健：标记医学图像以检测疾病或监测患者健康状况
- 环境：标记卫星图像以监测森林砍伐、提供野外管理信息或检测野火
- 农业：标记农作物图像以监测植物健康或用于土地使用监测的卫星图像
- 生态学：标记动物或植物物种的图像以监测野生动物种群或跟踪濒危物种

模型主页：https://huggingface.co/google/vit-base-patch16-224

In [48]:
from transformers import pipeline

classifier = pipeline(task="image-classification",model='apple/mobilevit-small')

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use mps:0


In [49]:
preds = classifier(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
)
preds = [{"score": round(pred["score"], 4), "label": pred["label"]} for pred in preds]
print(*preds, sep="\n")

{'score': 0.9441, 'label': 'lynx, catamount'}
{'score': 0.0041, 'label': 'tiger cat'}
{'score': 0.0036, 'label': 'grey fox, gray fox, Urocyon cinereoargenteus'}
{'score': 0.0026, 'label': 'Egyptian cat'}
{'score': 0.002, 'label': 'tabby, tabby cat'}


In [50]:
# 使用本地图片（狼猫）
preds = classifier(
    "data/image/cat-chonk.jpeg"
)
preds = [{"score": round(pred["score"], 4), "label": pred["label"]} for pred in preds]
print(*preds, sep="\n")

{'score': 0.9441, 'label': 'lynx, catamount'}
{'score': 0.0041, 'label': 'tiger cat'}
{'score': 0.0036, 'label': 'grey fox, gray fox, Urocyon cinereoargenteus'}
{'score': 0.0026, 'label': 'Egyptian cat'}
{'score': 0.002, 'label': 'tabby, tabby cat'}


![](data/image/cat-chonk.jpeg)

![](data/image/cat-chonk.jpg)

In [51]:
# 使用本地图片（熊猫）
preds = classifier(
    "data/image/panda.jpg"
)
preds = [{"score": round(pred["score"], 4), "label": pred["label"]} for pred in preds]
print(*preds, sep="\n")

{'score': 0.7917, 'label': 'giant panda, panda, panda bear, coon bear, Ailuropoda melanoleuca'}
{'score': 0.0047, 'label': 'lesser panda, red panda, panda, bear cat, cat bear, Ailurus fulgens'}
{'score': 0.0015, 'label': 'soccer ball'}
{'score': 0.0013, 'label': 'indri, indris, Indri indri, Indri brevicaudatus'}
{'score': 0.0012, 'label': 'Eskimo dog, husky'}


[](data/image/panda.jpeg)

[](data/image/panda.jpg)

### Object Detection

与图像分类不同，目标检测在图像中识别多个对象以及这些对象在图像中的位置（由边界框定义）。目标检测的一些示例应用包括：

- 自动驾驶车辆：检测日常交通对象，如其他车辆、行人和红绿灯
- 遥感：灾害监测、城市规划和天气预报
- 缺陷检测：检测建筑物中的裂缝或结构损坏，以及制造业产品缺陷

模型主页：https://huggingface.co/facebook/detr-resnet-50

#### 前置依赖包安装

In [ ]:
%pip install timm

Note: you may need to restart the kernel to use updated packages.


In [52]:
from transformers import pipeline

detector = pipeline(task="object-detection",model='facebook/detr-resnet-101-dc5')

Some weights of the model checkpoint at facebook/detr-resnet-101-dc5 were not used when initializing DetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
The `max_size` parameter is deprecated and will be removed in v4.26. Please spe

In [53]:
preds = detector(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
)
preds = [{"score": round(pred["score"], 4), "label": pred["label"], "box": pred["box"]} for pred in preds]
preds

[{'score': 0.9753,
  'label': 'cat',
  'box': {'xmin': 179, 'ymin': 154, 'xmax': 875, 'ymax': 596}}]

![](data/image/cat_dog.jpg)

### Homework：替换以上示例中的模型，对比不同模型在相同任务上的性能表现

在 Hugging Face Models 中找到适合你的模型：https://huggingface.co/models

In [54]:
preds = detector(
    "data/image/cat_dog.jpg"
)
preds = [{"score": round(pred["score"], 4), "label": pred["label"], "box": pred["box"]} for pred in preds]
preds

[{'score': 0.9994,
  'label': 'cat',
  'box': {'xmin': 76, 'ymin': 58, 'xmax': 317, 'ymax': 371}},
 {'score': 0.9986,
  'label': 'dog',
  'box': {'xmin': 284, 'ymin': 20, 'xmax': 484, 'ymax': 416}}]

In [ ]:
System